# Домашнее задание по теме «Линейная регрессия»

В этом задании ты обучишь линейную регрессию, применишь регуляризацию, нормировку, проанализируешь качество модели, а также реализуешь градиентный спуск.

## Подготовка

In [ ]:
import gc

import pandas as pd
import numpy as np

import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

import time
from tqdm import tqdm

from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import MinMaxScaler, StandardScaler

##  Задача 1 [0,5 балла]

Объясни, почему при использовании регуляризации штрафуются только веса, но не свободный коэффициент $w_0$ (другое обозначение — $b$)?

Напомним, что предсказание линейной модели считается по формуле:

$$a(x) = \langle w, x \rangle + w_0.$$

**Напиши ответ здесь:**

-------

Для следующих заданий возьми данные из датасета [Predict Calorie Expenditure](https://www.kaggle.com/competitions/playground-series-s5e5/overview).

Цель соревнования — предсказать, сколько калорий потратит человек за тренировку по признаковому описанию как человека, так и самой тренировки.

In [ ]:
! gdown 1jb1tbxmGcgFk-jzNgu-sHXCUkgtAslRq

In [ ]:
data = pd.read_csv("/content/train.csv", index_col=0)
data.head()

В таблице 8 столбцов: категориальный признак `Sex`, 6 числовых признаков и целевая переменная `Calories`.

Разделим данные на признаки и таргет, затем на `train` и `val`. В этой работе `val` используется для сравнения и выбора вариантов модели. Отдельный итоговый тест здесь не выделяем.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(data.drop('Calories', axis=1), data['Calories'], test_size=0.25, random_state=123)

## Задача 2. OneHotEncoder [0,5 балла]

В семинаре мы использовали `Target Encoding` для приведения категориальных переменных к числовому виду, так как в задаче было очень много категорий. Но для линейных моделей обычно используется `OneHotEncoding`.

Примени [OneHotEncoding из sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) к категориальной переменной и получи новый датасет, который уже будет использоваться для дальнейшего обучения моделей. Но не делай этого в явном виде. Задай последовательность действий в виде [пайплайна из sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html). Он тебе пригодится в задачах.

Затраты по калориям не бывают отрицательными. Модель этого не знает, так как она предсказывает на всей числовой прямой. Поэтому нужно избежать отрицательных значений на этапе вызова `predict` с помощью [TransformedTargetRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.compose.TransformedTargetRegressor.html).

Сделать неотрицательными предсказания калорий можно по-разному: округлить до нуля, взять минимальное по обучающей выборке.

**Задание**:
1. Преобразуй датасет к числовому виду.
2. Обучи модель линейной регрессии и сделай предсказание на валидации, чтобы проверить, что всё работает.
3. Добавь постобработку ответов.
4. Оберни все в Pipeline.

In [ ]:
# Импортируем нужные модули
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline

# Найди минимальный y
# Напиши код здесь

# Внутри Pipeline преобразуй категориальные переменные и передай результат линейной модели
pipe = # Напиши код здесь

# Обучи модель и получи предсказания для валидации, чтобы проверить работоспособность
# Напиши код здесь

y_pred_val = pipe.
y_pred_val.shape, (y_pred_val < 0).sum()

##  Задача 3. Добавь функцию оценки качества работы [0,5 балла]

В соревновании используется не самая стандартная метрика RMSLE (англ. Root Mean Squared Logarithmic Error).

$$RMSLE = \Big(\frac{1}{n}\sum_{i=1}^n(\log{(1+\hat y_i)} - \log{(1+y_i)})^{2}\Big)^{\frac{1}{2}}$$

Эта метрика есть в реализации [sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_log_error.html), но тебе нужно написать её самостоятельно, чтобы в следующих заданиях её можно было передавать внутрь пайплайна через [make_scorer](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.make_scorer.html).

Подумай, с каким знаком должна возвращать ответ эта функция для корректной работы `make_scorer`.

**Задание:**
1. Реализуй функцию для подсчёта метрики.
2. Проверь, что она работает корректно (сравни с примером из sklearn).
3. Оцени качество предсказаний на `train` и `val`.
4. Занеси положительную RMSLE в таблицу с историей. Используй единое имя `scoring_foo(y_true, y_pred)` для скоринговой функции во всех следующих заданиях. В `make_scorer` используется соглашение «больше — лучше», а в таблице RMSLE «меньше — лучше».

In [ ]:
from sklearn.metrics import root_mean_squared_log_error

def scoring_foo(y_true, y_pred): # Это имя используется и в следующих заданиях
  # Напиши код здесь
  return

# Получи предсказания на train
y_pred_train = pipe. # Напиши код здесь

# Сравни ответы для валидации
print(scoring_foo(y_val, y_pred_val))
print(root_mean_squared_log_error(y_val, y_pred_val))

# Получи оценку метрики для train и val
rmsle_train = -scoring_foo(y_train, y_pred_train)
rmsle_valid = -scoring_foo(y_val, y_pred_val)

# Создай контейнер для хранения информации о метриках
res = {}

# Запиши метрики в контейнер и выведи таблицу
res['base'] = [rmsle_train, rmsle_valid]
df_res = pd.DataFrame.from_dict(res, orient='index',
                                columns=['rmsle_train', 'rmsle_valid'])
df_res

##  Задача 4. Выбери модель и нормализацию [1 балл]

Используй модели:
* Ridge,
* Lasso,
* ElasticNet.

Для их качественной работы примени нормировки:
* MinMaxScaler,
* StandardScaler,
* RobustScaler.

**Задание.**
Оцени качество работы каждой пары нормировка — модель и занеси результаты в таблицу с соответствующим названием. По названию должно быть понятно, к какой комбинации относится результат.

Нормировку добавь внутрь пайплайна. За основу возьми пайплайн из задания выше.

In [ ]:
from sklearn.linear_model import Ridge, ElasticNet, Lasso
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler

# Напиши код здесь

# Выведи результат
df_res

## Задача 5. Добавь полиномы [1 балл]



Линейная модель может решать нелинейные задачи, если к признаковому пространству добавить их полиномы. Вряд ли сожжённые за тренировку калории линейно зависят от признаков. Поэтому добавь полиномы различных степеней к признаковому пространству, чтобы улучшить качество решения задачи.

В этом тебе поможет класс [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html), он также интегрируется в пайплайн решения. Подумай, куда конкретно его вставлять в пайплайн и нужно ли добавлять полиномы OHE переменных.

Обязательно изучи документацию: там есть один аргумент, который тебе надо выключить.

Для добавления полиномов руководствуйся логикой или общими знаниями. Например, в экономических и социологических исследованиях часто добавляют к данным переменную «квадрат возраста». Её логика в следующем: человек растёт и развивается до определённого возраста, а далее угасает. Скорее всего, целевая переменная и возраст будут зависеть параболически, «квадрат возраста» позволит учесть эту параболическую зависимость.

Если не можешь придумать полиномы, то перебери переменные и значения полиномов для них. Переменных не так много, а модель учится быстро.

**Задание:**
1. Возьми лучшую по валидации комбинацию нормировка — модель.
2. Добавь к пайплайну полиномы для некоторых переменных.
3. Сохрани результат лучшего эксперимента в таблицу (у авторов задания получилось качество 0.238208).
4. Опиши, как выбирал полиномы.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Напиши код здесь

**Опиши, как ты подбирал полиномы здесь:**

##  Задача 6. Добавь дискретизацию [1 балл]

Некоторые вещественные признаки можно разбить на интервалы или дискретизовать. Для этого в sklearn есть функционал [KBinsDiscretizer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.KBinsDiscretizer.html). Примени его к переменным по своему усмотрению, чтобы улучшить качество.

**Задание:**
1. Добавь `KBinsDiscretizer` в пайплайн. Документация поможет тебе выбрать правильные аргументы.
2. Вынеси лучшее качество в таблицу с результатам (у авторов задания получилось 0.234339).

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

# Напиши код здесь

## Задача 7. Кросс-валидация и усреднение прогнозов [0,5 балла]

Обучим пять моделей на фолдах с помощью [cross_validate](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html). Передай `cv=5`, `return_estimator=True` и скоринговую функцию из задачи 3 через `make_scorer`.

Здесь два разных действия:

1. Кросс-валидация оценивает качество на отложенных фолдах. Выведи среднее RMSLE по фолдам из `test_score`, вернув положительный знак метрики.
2. Усреднение прогнозов пяти обученных моделей создаёт ансамбль. Посчитай его прогнозы на `train` и `val`, добавь RMSLE в таблицу и сравни с одной моделью из задачи 6.

Усреднённый прогноз ансамбля на `train` не является out-of-fold-прогнозом: для каждой строки четыре из пяти моделей обучались на ней. Не используй эту train-метрику как независимую оценку качества. `val` здесь используется для выбора вариантов модели, поэтому её результаты также не являются оценкой на отдельном итоговом тесте.

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer

cv_results = # Напиши код здесь

# Выведи среднее RMSLE по отложенным фолдам из cv_results["test_score"]
# Напиши код здесь

# Функция для получения предсказаний по фолдам
def get_predictions(cv_results, X):
  # Напиши код здесь
  pass

# Получи предсказания
y_pred_train = get_predictions(cv_results, X_train)
y_pred_val = get_predictions(cv_results, X_val)

# Получи оценки метрики для train и val
rmsle_train = -scoring_foo(y_train, y_pred_train)
rmsle_valid = -scoring_foo(y_val, y_pred_val)

# Запиши метрики в контейнер и выведи таблицу
res['general_cv'] = [rmsle_train, rmsle_valid]
df_res = pd.DataFrame.from_dict(res, orient='index',
                                columns=['rmsle_train', 'rmsle_valid'])

# Выведи результат
df_res

Получилось ли улучшить результат за счёт усреднения прогнозов? Сравни `general_cv` и `best_feat_desc` по `rmsle_valid` и объясни результат. Чем это усреднение отличается от оценки качества по кросс-валидации?

**Напиши ответ здесь:**

## Задача 8. Реализуй градиентный спуск для MSE [3 балла]

Реализуем итеративное обучение линейной модели, чтобы разобраться, как обновляются её коэффициенты. При небольшом числе признаков аналитическое решение доступно и в Colab. Здесь градиентный спуск нужен как учебный эксперимент.

**Задание:**

1. Возьми предобработку из задачи 6. Обучи её только на `train` с помощью `fit_transform`, а к `val` примени `transform`.
2. Собери матрицу «объекты × признаки» и добавь первым столбец единиц для свободного коэффициента $w_0$.
3. Выведи градиент MSE в LaTeX или на бумаге и приложи фото [1 балл].
4. Для сравнения со sklearn используй Lasso с `alpha=1.0` на тех же преобразованных признаках. В sklearn минимизируется половина MSE плюс L1-штраф:

   $$J(w)=\frac{1}{2n}\|Xw-y\|_2^2+\alpha\sum_{j=1}^{p}|w_j|.$$

   Реализуй `mse_grad` для этой функции. Свободный коэффициент $w_0$ не штрафуется. Для L1 используй субградиент `np.sign(w[j])`, в нуле выбери 0 [1 балл]. Субградиент заменяет производную в точке излома.
5. Реализуй `gradient_descent`, который на каждой итерации вызывает переданную функцию `grad_foo`. Параметры описаны в заготовке [1 балл].
6. Получи прогнозы на `train` и `val`, примени ту же постобработку, что выше, и посчитай RMSLE.
7. Сравни с Lasso: отдельно значение $J$ на обучении и RMSLE на валидации. Для $J$ используй исходные линейные прогнозы, до постобработки. При достаточном числе итераций ожидаем близкие значения $J$. Более низкое $J$ не обязано давать более низкую RMSLE. При заметном расхождении проверь нормировку, штраф, шаг и число итераций.

Из-за L1-штрафа и постоянного шага субградиентный метод может колебаться около минимума. Достижение лимита итераций само по себе не означает ошибку в формуле.

**Напиши формулу матричного градиента для MSE здесь:**

In [ ]:
# Используй предобработку из задачи 6
prep = pipe[:-1]

def add_bias(X):
    # Добавь первым столбец единиц
    pass

X_train_b = add_bias(prep.transform(X_train))
X_val_b = add_bias(prep.transform(X_val))
alpha = 1.0

# Половина MSE + L1. Не штрафуй свободный член.
def mse_grad(w, X, y):
    # Напиши код здесь
    pass


def gradient_descent(X, y, grad_foo, lr=0.1, n_iter=5000, tol=1e-4, w_init=None):
    """
    grad_foo: функция (w, X, y), возвращающая градиент или субградиент.
    lr: шаг обучения, n_iter: максимальное число обновлений.
    tol: порог нормы градиента для досрочной остановки.
    w_init: начальные веса. Если None, задай прогноз средним y:
        все наклоны равны нулю, свободный член равен np.mean(y).
    Если массив передан, скопируй его, чтобы не изменить предыдущие веса.
    """
    # Напиши код здесь. Используй grad_foo и np.linalg.norm.
    pass


In [ ]:
# Получи веса для MSE
w_gd_mse = gradient_descent(X_train_b, y_train.values, mse_grad, lr=0.1, n_iter=5000, tol=1e-4)

# Функция для получения предсказаний
def predict(X_b, w):
    return # Напиши код здесь

# Функция для постобработки ответов. Всё, что меньше минимального значения в датасете, увеличь до минимального
def postprocessing(y_pred):
  return np.maximum(y_pred, y_min)

# Получи предсказания
y_pred_train = postprocessing(predict(X_train_b, w_gd_mse))
y_pred_val = postprocessing(predict(X_val_b, w_gd_mse))

# Получи оценки метрики для train и val
rmsle_train = -scoring_foo(y_train, y_pred_train)
rmsle_valid = -scoring_foo(y_val, y_pred_val)

# Запиши метрики в контейнер и выведи таблицу
res['best_handmade'] = [rmsle_train, rmsle_valid]
df_res = pd.DataFrame.from_dict(res, orient='index',
                                columns=['rmsle_train', 'rmsle_valid'])

# Выведи результат
df_res

# Сравни с Lasso(alpha=alpha, max_iter=10000, tol=1e-6) на тех же признаках.
# Выведи J на train до постобработки и RMSLE на val после неё.
# Напиши код здесь


## Задача 9. Реализуй градиентный спуск для RMSLE [2 балла]

Теперь изменим функцию ошибки при обучении. У MSE и RMSLE в общем случае разные минимумы: RMSLE сравнивает ответы в логарифмической шкале. Например, для двух ответов 1 и 9 лучшая константа по MSE равна 5, а по RMSLE примерно 3,47.

Сохраним линейный прогноз $z=Xw$ и постобработку $\hat y_i=\max(y_{min},z_i)$ из предыдущих задач. **Вычисляй RMSLE и её градиент с учётом этой постобработки на каждом шаге обучения**, чтобы логарифм получал допустимые значения.

Для $z_i>y_{min}$ производная прогноза по $w$ равна $x_i$, а для $z_i<y_{min}$ равна нулю. В точке $z_i=y_{min}$ выберем нулевую производную. При нулевой RMSLE также верни нулевой вклад ошибки в градиент, чтобы не делить на ноль. В остальных точках используй правило дифференцирования сложной функции.

Минимизируем

$$J_{log}(w)=RMSLE(y,\hat y)+\alpha_{log}\sum_{j=1}^{p}|w_j|,\qquad \alpha_{log}=10^{-4}.$$

Масштаб RMSLE намного меньше масштаба MSE, поэтому силу штрафа задаём отдельно. Свободный член не штрафуем. Это зафиксированные настройки учебного опыта, без обещания оптимального качества.

**Задание:**

1. Выведи формулу градиента RMSLE с учётом постобработки в LaTeX или на бумаге [1 балл].
2. Реализуй `rmsle_grad` и добавь L1-штраф с коэффициентом `alpha_log` [1 балл].
3. Начни обучение с коэффициентов `w_gd_mse` из задачи 8: передай их через `w_init`, не изменяя исходный массив. Используй тот же алгоритм спуска с новой функцией градиента.
4. Получи прогнозы и сравни RMSLE до и после дообучения. Объясни результат. Улучшение на валидации не гарантируется.

После отсечения функция имеет изломы и плоские участки. Мы используем описанное правило обновления в этих точках, а не предполагаем дифференцируемость всюду.

**Напиши формулу матричного градиента для RMSLE здесь:**

In [ ]:
# RMSLE после постобработки + L1. Свободный член не штрафуем.
alpha_log = 1e-4
def rmsle_grad(w, X, y):
  # Напиши код здесь
  pass

# Получи веса
w_gd_rmsle = gradient_descent(X_train_b, y_train.values, rmsle_grad, lr=1.0, n_iter=5000, tol=1e-5, w_init=w_gd_mse)

# Выведи предсказания
y_pred_train = postprocessing(predict(X_train_b, w_gd_rmsle))
y_pred_val = postprocessing(predict(X_val_b, w_gd_rmsle))

# Получи оценку метрики для train и val
rmsle_train = -scoring_foo(y_train, y_pred_train)
rmsle_valid = -scoring_foo(y_val, y_pred_val)

res['best_handmade_rmsle'] = [rmsle_train, rmsle_valid]
df_res = pd.DataFrame.from_dict(res, orient='index',
                                columns=['rmsle_train', 'rmsle_valid'])

# Выведи результат
df_res